### This notebook can be used to generate the desired k range lookup table for Obliqua. 

In [2]:
import Pkg; Pkg.add("NCDatasets")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed aws_c_cal_jll ───────── v0.9.13+0
   Installed aws_c_sdkutils_jll ──── v0.2.4+1
   Installed aws_c_auth_jll ──────── v0.9.6+0
   Installed MPIPreferences ──────── v0.1.12
   Installed Hwloc_jll ───────────── v2.13.0+1
   Installed MPIABI_jll ──────────── v0.1.4+0
   Installed NetCDF_jll ──────────── v401.1000.0+0
   Installed CFTime ──────────────── v0.2.8
   Installed aws_c_common_jll ────── v0.12.6+0
   Installed aws_c_io_jll ────────── v0.26.3+0
   Installed s2n_tls_jll ─────────── v1.7.2+0
   Installed OpenMPI_jll ─────────── v5.0.11+0
   Installed MPICH_jll ───────────── v5.0.1+0
   Installed aws_c_http_jll ──────── v0.10.13+0
   Installed NCDatasets ──────────── v0.14.15
   Installed HDF5_jll ────────────── v2.1.2+0
   Installed aws_checksums_jll ───── v0.2.10+0
   Installed aws_c_compression_jll ─ v0.3.2+0
   Installed libaec_jll ──────────── v1.1.6+0
   Installed aws_c_s3_j

In [13]:
using NCDatasets

# Get Obliqua root directory
ROOT_DIR = abspath(joinpath(dirname(abspath(@__FILE__)),"../"))
RES_DIR  = joinpath(ROOT_DIR,"res/")

include(joinpath(ROOT_DIR, "src/Hansen.jl"))
import .Hansen

In [ ]:
# Define parameters
n_list = [2, 3, 4]     # List of tidal degrees 'n' to consider
ecc_list = range(0.0, stop=0.89, step=0.01) |> collect

# One may wish to adjust the threshold value, which represents the minimum absolute value 
# of Hansen coefficients to be considered significant. A lower threshold will include more 
# k values, while a higher threshold will be more selective. The choice of threshold may depend 
# on the specific application and the desired balance between accuracy and computational efficiency.
threshold = 1e-3       # Threshold for significant Hansen coefficients
k_min_global = -500    # Minimum k to consider globally
k_max_global = 500     # Maximum k to consider globally

# Prepare arrays (use Julia vectors)
n_vals    = Int32[]    # Tidal degree 'n'
m_vals    = Int32[]    # Tidal order 'm'
ecc_vals  = Float64[]  # Eccentricity
kmin_vals = Int32[]    # Minimum k values
kmax_vals = Int32[]    # Maximum k values

# Loop over n, m, and ecc
for n in n_list
    for m in 0:n
        for e in ecc_list
            k, X = Hansen.get_hansen(e, n, m, k_min_global, k_max_global)

            mask = abs.(X) .>= threshold
            k_sig = k[mask]

            if isempty(k_sig)
                kmin, kmax = 0, 0
            else
                kmin, kmax = minimum(k_sig), maximum(k_sig)
            end

            push!(n_vals, Int32(n))
            push!(m_vals, Int32(m))
            push!(ecc_vals, Float64(e))
            push!(kmin_vals, Int32(kmin))
            push!(kmax_vals, Int32(kmax))
        end
    end
end

# Create NetCDF file
ds = NCDataset(joinpath(RES_DIR, "hansen_k_table.nc"), "c")

# Dimension
defDim(ds, "entry", length(n_vals))

# Variables
n_var    = defVar(ds, "n", Int32, ("entry",))
m_var    = defVar(ds, "m", Int32, ("entry",))
ecc_var  = defVar(ds, "ecc", Float64, ("entry",))
kmin_var = defVar(ds, "k_min", Int32, ("entry",))
kmax_var = defVar(ds, "k_max", Int32, ("entry",))

# Assign data
n_var[:]    = n_vals
m_var[:]    = m_vals
ecc_var[:]  = ecc_vals
kmin_var[:] = kmin_vals
kmax_var[:] = kmax_vals

close(ds)

println("NetCDF file '$(joinpath(RES_DIR, "hansen_k_table.nc"))' created successfully.")

NetCDF file 'hansen_k_table.nc' created successfully.


### Obliqua can access this table to determine the k range for Hansen coefficients based on eccentricity and tidal degree/order, through the `get_k_range` function defined in `Hansen.jl`.

In [23]:
Hansen.get_k_range(0.5, 2, 2)  # Example usage

(-5, 27)